# Fundamentos de OpenCV

## ¿Qué es OpenCV?

**OpenCV** (Open Source Computer Vision Library) es la librería más ampliamente usada para visión por computador y procesamiento de imágenes.

| Característica | Descripción |
|---------|-------------|
| **Lenguaje** | C++ con bindings para Python |
| **Licencia** | BSD (libre para uso comercial) |
| **Funciones** | Más de 2500 algoritmos optimizados |
| **Aplicaciones** | Procesamiento de imágenes, detección de objetos, reconocimiento facial, análisis de vídeo |

OpenCV se usa frecuentemente como **herramienta de preprocesamiento** para el aprendizaje profundo (carga, transformación y aumento de imágenes) y para algoritmos de **visión por computador clásica**.
Puedes instalarlo con el paquete [opencv-python](https://pypi.org/project/opencv-python/).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
print(f"Versión de OpenCV: {cv2.__version__}")

## Cargar y Mostrar Imágenes

OpenCV usa el orden de color **BGR** por defecto (no RGB como matplotlib o PyTorch). Esto es una peculiaridad histórica: el hardware de cámara antiguo y la librería Intel original almacenaban los bytes en ese orden, y OpenCV lo mantuvo por compatibilidad.

> **Regla práctica:** siempre que pases una imagen de OpenCV a matplotlib o a una red neuronal, conviértela primero con `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`.

In [ ]:
sample_path = "../resources/images/lenna.png"
img = cv2.imread(sample_path) # Cargar imagen (en BGR)
print(f"Forma de la imagen: {img.shape}")  # (alto, ancho, canales (BGR))
print(f"Tipo de la imagen: {img.dtype}")  # uint8 (0-255); intensidad del color en el píxel

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img)  # asume RGB
axes[0].set_title("Sin conversión (BGR→interpretado como RGB)")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # convertir a RGB
axes[1].imshow(img_rgb)
axes[1].set_title("Con cv2.cvtColor (correcto)")
plt.show()

### Las Imágenes son Arrays de `uint8`

`uint8` significa **entero sin signo de 8 bits**: valores en el rango [0, 255].
- 0 = sin intensidad (negro en escala de grises, sin contribución para un canal de color)
- 255 = intensidad máxima (blanco / canal saturado)

¿Por qué 8 bits por canal? Un byte por canal es un compromiso práctico: suficientes niveles (256 por canal = 16,7 M de colores posibles en RGB) para evitar el bandeo visible al ojo humano, sin requerir más memoria.

Este rango importa de dos maneras prácticas:
1. **Recorte**: los valores que caen fuera del rango de imagen válido deben forzarse de vuelta a `[0, 255]`. Por ejemplo, si una operación produce 300, el recorte lo convierte en 255.
2. **Desbordamiento**: la aritmética hecha directamente en `uint8` puede envolver antes de recortar. `np.uint8(200) + np.uint8(100)` se convierte en 44, no en 300, porque la suma se calcula módulo 256.
3. **Normalización para redes neuronales**: [CLIP](https://openai.com/index/clip/) y la mayoría de los modelos de torchvision esperan valores en `[0, 1]` (o normalizados con media/std). Dividir por `255.0` convierte desde el rango `uint8` de OpenCV.

Un flujo de trabajo seguro es: convertir a `float32` → hacer la aritmética → recortar a `[0, 255]` → volver a `uint8`.

In [ ]:
# Crear una imagen de prueba simple (gradiente)
height, width = 200, 300
img_canvas = np.zeros((height, width, 3), dtype=np.uint8) # array de 3 canales
# Crear gradiente RGB (R aumenta de izquierda a derecha, G aumenta de arriba a abajo)
for y in range(height):
    for x in range(width):
        img_canvas[y, x] = [int(255 * x / width), # Rojo
                     int(255 * y / width), # Verde
                     100] # Azul

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_canvas)  # asume RGB
axes[0].set_title("RGB")
img_rgb = cv2.cvtColor(img_canvas, cv2.COLOR_BGR2RGB)
axes[1].imshow(img_rgb)
axes[1].set_title("BGR")
plt.tight_layout()
plt.show()

In [ ]:
# Diferentes modos de carga.
# cv2.IMREAD_COLOR     (por defecto, BGR de 3 canales)
# cv2.IMREAD_GRAYSCALE (1 canal)
# cv2.IMREAD_UNCHANGED (mantiene el canal alfa si está presente, p. ej. para PNG con transparencia)
img_color = cv2.imread(sample_path, cv2.IMREAD_COLOR)
img_gray = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
img_unchanged = cv2.imread(sample_path, cv2.IMREAD_UNCHANGED)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Color: {img_color.shape}")
axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title(f"Escala de grises: {img_gray.shape}")
axes[2].imshow(cv2.cvtColor(img_unchanged, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Sin cambios: {img_unchanged.shape}")
plt.tight_layout()
plt.show()

## Espacios de Color

OpenCV soporta muchas conversiones de espacio de color:

| Espacio de Color | Caso de Uso |
|-------------|----------|
| **BGR** | Formato predeterminado de OpenCV |
| **RGB** | Visualización, aprendizaje profundo |
| **Escala de grises** | Detección de bordes, umbralización |
| **HSV** | Segmentación basada en color |
| **LAB** | Espacio de color perceptualmente uniforme |

In [ ]:
# Convertir a diferentes espacios de color
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
# Canales RGB
for i, (name, channel) in enumerate([('R', img_rgb[:,:,0]), ('G', img_rgb[:,:,1]), ('B', img_rgb[:,:,2])]):
    axes[0, i].imshow(channel, cmap='gray')
    axes[0, i].set_title(f"Canal {name}")
# Canales HSV
for i, (name, channel) in enumerate([('Matiz', img_hsv[:,:,0]), ('Saturación', img_hsv[:,:,1]), ('Valor', img_hsv[:,:,2])]):
    axes[1, i].imshow(channel, cmap='gray')
    axes[1, i].set_title(f"{name}")
plt.tight_layout()
plt.show()

## Transformaciones Básicas

### Redimensionar

### Redimensionado e Interpolación

Redimensionar cambia el número de píxeles en la imagen. La decisión clave no es solo el nuevo tamaño, sino también **cómo OpenCV inventa o fusiona valores de píxeles** durante ese cambio.

Opciones comunes de interpolación:
- `INTER_NEAREST`: copia el píxel más cercano. El más rápido, pero con bloques al ampliar.
- `INTER_LINEAR`: mezcla píxeles cercanos. Buen valor predeterminado para la mayoría del redimensionado.
- `INTER_CUBIC`: usa un vecindario más amplio. Más lento, pero a menudo más suave para ampliar.
- `INTER_AREA`: promedia píxeles. Generalmente la mejor opción para reducir porque preserva la estructura general y reduce el aliasing.

Si una imagen ampliada parece pixelada, generalmente no es un error en `cv2.resize`; significa que el método mantuvo los bordes de píxeles duros visibles. Eso es exactamente lo que hace `INTER_NEAREST`, y es útil cuando se quieren preservar píxeles discretos, como en máscaras de segmentación o pixel art.

También recuerda que `cv2.resize` espera el tamaño como `(ancho, alto)`, que es el orden inverso a la forma NumPy de una imagen `(alto, ancho, canales)`.

In [ ]:
# Redimensionar especificando dimensiones.
# Nota: cv2.resize toma (ancho, alto) — el orden inverso al (filas, columnas) de NumPy.
img_small = cv2.resize(img, (100, 100))
# Redimensionar por factor de escala
img_half = cv2.resize(img, None, fx=0.5, fy=0.5)
# Los métodos de interpolación controlan calidad vs velocidad al cambiar la resolución.
# INTER_NEAREST — más rápido; bueno para miniaturas donde la calidad no importa.
# INTER_LINEAR  — bilineal; predeterminado; buen equilibrio para la mayoría de los casos.
# INTER_CUBIC   — bicúbico; escalado más nítido pero ~4× más lento que LINEAR.
# INTER_AREA    — usar al REDUCIR; evita artefactos de moiré.
img_nearest = cv2.resize(img, (100, 100), interpolation=cv2.INTER_NEAREST)
img_linear = cv2.resize(img, (100, 100), interpolation=cv2.INTER_LINEAR)
img_cubic = cv2.resize(img, (100, 100), interpolation=cv2.INTER_CUBIC)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, im) in zip(axes, [('NEAREST', img_nearest), ('LINEAR', img_linear), ('CUBIC', img_cubic)]):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{name}: {im.shape[:2]}")
plt.tight_layout()
plt.show()

### Rotación y Volteo

In [ ]:
# Volteo — código de volteo: 1=horizontal, 0=vertical, -1=ambos ejes
img_flip_h = cv2.flip(img, 1)
img_flip_v = cv2.flip(img, 0)
img_flip_both = cv2.flip(img, -1)

# La rotación requiere una matriz de transformación afín (2×3).
# getRotationMatrix2D(centro, ángulo_grados, escala):
#   el ángulo es en sentido antihorario; escala=1.0 mantiene el mismo tamaño.
h, w = img.shape[:2]
center = (w // 2, h // 2)
M = cv2.getRotationMatrix2D(center, 45, 1.0)
# warpAffine aplica M a cada píxel.
# El tamaño de salida (w, h) se mantiene igual — las esquinas que caen fuera se recortan.
img_rotated = cv2.warpAffine(img, M, (w, h))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, im) in zip(axes, [('Volteo H', img_flip_h), ('Volteo V', img_flip_v),
                                   ('Ambos', img_flip_both), ('Rotar 45°', img_rotated)]):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Recorte

In [ ]:
# El recorte usa slicing de NumPy
h, w = img.shape[:2]
# Recortar la región central
margin = 100
img_cropped = img[margin:h-margin, margin:w-margin]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original: {img.shape[:2]}")
axes[1].imshow(cv2.cvtColor(img_cropped, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Recortada: {img_cropped.shape[:2]}")
plt.tight_layout()
plt.show()

## Dibujar sobre Imágenes

OpenCV proporciona funciones para dibujar formas y texto sobre imágenes.

In [ ]:
canvas = img.copy()
# Todas las funciones de dibujo usan tuplas de color BGR y modifican la imagen en el lugar.
# rectangle(img, pt1_esquina_sup_izq, pt2_esquina_inf_der, color_bgr, grosor)
cv2.rectangle(canvas, (50, 50), (200, 200), (0, 255, 0), 3)
# circle(img, centro, radio, color_bgr, grosor)
# grosor=-1 rellena la forma.
cv2.circle(canvas, (350, 150), 80, (0, 0, 255), -1)
# line(img, pt1, pt2, color_bgr, grosor)
cv2.line(canvas, (400, 50), (500, 200), (255, 0, 0), 5)
# putText(img, texto, org_abajo_izq, fuente, escalaFuente, color, grosor)
cv2.putText(canvas, "OpenCV", (50, 300), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 3)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("Dibujar sobre imágenes")
plt.show()

## Guardar Imágenes

**JPEG vs PNG:**
- **JPEG** es con pérdida: descarta información para lograr archivos más pequeños (controlado por la calidad 0–100). Con calidad 95 la diferencia es invisible para el ojo; con calidad 50 aparecen artefactos alrededor de los bordes nítidos. No uses JPEG para máscaras de etiquetas, imágenes de anotación ni ninguna imagen que vayas a procesar más.
- **PNG** es sin pérdida: cada píxel se almacena exactamente. Los archivos son más grandes, pero no se pierde información. Usa PNG siempre que la fidelidad sea más importante que el tamaño del archivo.

In [ ]:
import os
output_dir = "../artifacts/outputs"
os.makedirs(output_dir, exist_ok=True)

# Guardar en diferentes formatos
output_jpg = os.path.join(output_dir, "output.jpg")
output_png = os.path.join(output_dir, "output.png")
output_low = os.path.join(output_dir, "output_low.jpg")
cv2.imwrite(output_jpg, img)  # JPEG (con pérdida)
cv2.imwrite(output_png, img)  # PNG (sin pérdida)
# Parámetro de calidad JPEG (0-100)
cv2.imwrite(output_low, img, [cv2.IMWRITE_JPEG_QUALITY, 50])
print("¡Imágenes guardadas!")

# Limpieza
for path in [output_jpg, output_png, output_low]:
    if os.path.exists(path):
        os.remove(path)

## Integración con PyTorch

OpenCV se usa habitualmente para cargar y preprocesar imágenes antes de alimentarlas a redes neuronales.
Existe una conexión directa con las CNN, pero los roles son diferentes:
- En el **procesamiento de imágenes clásico**, elegimos el filtro a mano. Por ejemplo, un kernel de detección de bordes está diseñado por nosotros porque ya sabemos qué patrón queremos resaltar.
- En una **red neuronal convolucional**, la operación sigue siendo una convolución, pero los valores del kernel se **aprenden de los datos** durante el entrenamiento.

Así que muchos efectos que se ven en los notebooks de procesamiento de imágenes, como el desenfoque, el enfoque y la detección de bordes, se construyen a partir de la misma idea matemática utilizada en las capas convolucionales. La diferencia importante es que los filtros de OpenCV son herramientas fijas, mientras que los filtros de CNN son parámetros entrenables.

In [ ]:
import torch

def preprocess_for_pytorch(img_path, target_size=(224, 224)):
    """Cargar y preprocesar imagen para el modelo PyTorch."""
    # Cargar imagen
    img = cv2.imread(img_path)
    
    # Convertir BGR a RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Redimensionar
    img = cv2.resize(img, target_size)
    
    # Convertir a flotante y normalizar a [0, 1]
    img = img.astype(np.float32) / 255.0
    
    # Convertir a tensor PyTorch: (H, W, C) → (C, H, W)
    tensor = torch.from_numpy(img).permute(2, 0, 1)
    
    # Añadir dimensión de batch: (C, H, W) → (1, C, H, W)
    tensor = tensor.unsqueeze(0)
    
    return tensor

tensor = preprocess_for_pytorch(sample_path)
print(f"Forma del tensor: {tensor.shape}")  # (1, 3, 224, 224)
print(f"Tipo del tensor: {tensor.dtype}")
print(f"Rango de valores: [{tensor.min():.3f}, {tensor.max():.3f}]")

## Resumen

| Operación | Función |
|-----------|----------|
| Cargar imagen | `cv2.imread(path, flags)` |
| Guardar imagen | `cv2.imwrite(path, img)` |
| Conversión de color | `cv2.cvtColor(img, cv2.COLOR_*)` |
| Redimensionar | `cv2.resize(img, (w, h))` |
| Rotar | `cv2.warpAffine(img, M, (w, h))` |
| Voltear | `cv2.flip(img, code)` |
| Dibujar rectángulo | `cv2.rectangle(img, pt1, pt2, color, thickness)` |
| Dibujar círculo | `cv2.circle(img, center, radius, color, thickness)` |
| Dibujar texto | `cv2.putText(img, text, org, font, scale, color, thickness)` |